# Financial Sensitivity Dashboard

Sliders recalculate revenue and profit scenarios with charts and ranked impacts

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

df = pd.read_csv("data/software_companies_dataset_v2.csv")

# Prepare missing values for reliable widgets and Plotly charts.
categorical_columns = [
    "Company_Name", "Industry", "Headquarters_City", "Country",
    "Ownership_Type", "Customer_Segment", "Primary_Cloud", "Risk_Rating"
]
for column in categorical_columns:
    if column in df.columns:
        df[column] = df[column].fillna("Unknown").astype(str).str.strip()

numeric_columns = [
    "Employees", "Annual_Revenue", "Profit_Margin", "Market_Share",
    "R&D_Spending", "Average_Salary", "Training_Hours_Per_Employee",
    "Employee_Satisfaction", "Adoption_Rate_AI", "Adoption_Rate_Cloud",
    "Adoption_Rate_Blockchain"
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
        df[column] = df[column].fillna(df[column].median())

df["Annual_Revenue"] = df["Annual_Revenue"].clip(lower=1)
df["Employees"] = df["Employees"].clip(lower=1)

growth = widgets.IntSlider(value=8, min=-20, max=40, step=2, description="Growth %")
margin = widgets.FloatSlider(value=1.5, min=-10, max=10, step=.5, description="Margin Δ")
out = widgets.Output()

def render(*_):
    s = df.copy()
    s["Scenario_Revenue"] = s["Annual_Revenue"] * (1 + growth.value/100)
    s["Base_Profit"] = s["Annual_Revenue"] * s["Profit_Margin"]/100
    s["Scenario_Profit"] = s["Scenario_Revenue"] * (s["Profit_Margin"]+margin.value)/100
    s["Impact"] = s["Scenario_Profit"] - s["Base_Profit"]
    with out:
        clear_output(wait=True)
        display(widgets.HTML(f"<h3>Incremental profit: ${s.Impact.sum()/1e6:,.1f}M</h3>"))
        px.bar(s.nlargest(20,"Impact").sort_values("Impact"),
               x="Impact", y="Company_Name", orientation="h",
               title="Companies with largest scenario impact").show()

growth.observe(render, names="value")
margin.observe(render, names="value")
display(widgets.HBox([growth, margin]), out)
render()